# Preprocessing and annotation

Extract seqlets from contribution scores, score them against a motif collection, and store
everything in a single {class}`anndata.AnnData` object:

- Loading motif collections and annotations
- Extracting seqlets from contribution scores
- Calculating motif similarities
- Annotating each seqlet with a TF family
- Creating and saving TF-MInDi's AnnData objects

`tfmindi` auto-detects a GPU and uses it for every heavy step; here the TomTom motif similarity.
Override with `tm.set_backend("cpu")` or the `TFMINDI_BACKEND` environment variable; a GPU step that
fails falls back to CPU with a warning.

## Data requirements

Three inputs:

- a motif database of known transcription factor motifs
- one-hot encoded sequences for your regions of interest, shape `(n_regions, 4, width)`
- attribution (contribution) scores for those same regions, same shape

`tfmindi` can fetch SCENIC+ motif collections if you don't have your own. Producing attribution
scores is out of scope; we recommend [CREsted](https://crested.readthedocs.io/en/latest/api/tools/_autosummary/crested.tl.contribution_scores_specific.html)
or [tangermeme](https://tangermeme.readthedocs.io/en/latest/tutorials/Tutorial_A3_Deep_LIFT_SHAP.html),
but anything that gives you numpy arrays works.

## Download tutorial data

### Contribution scores

Contribution score data to reproduce this tutorial is available on Zenodo using doi: [10.5281/zenodo.18757793](https://doi.org/10.5281/zenodo.18757793). These are contribution scores generated using the [deepBICCN model](https://crested.readthedocs.io/en/latest/models/BICCN/deepbiccn2.html), see [CREsted tutorial for more info](https://crested.readthedocs.io/en/latest/tutorials/enhancer_code_analysis.html).

In [1]:
import os

if not os.path.exists("./tutorial_data"):
    !wget https://zenodo.org/records/18757794/files/tutorial_data.tar.gz
    !tar -xvf tutorial_data.tar.gz

## Fetching motif collections and annotations

Seqlets are scored against a reference motif collection with {func}`~tfmindi.pp.calculate_motif_similarity`.

TF-MInDi uses the **v11 Aerts lab motif collection**: ~70k motifs from many source databases
(JASPAR, Cis-BP, HOCOMOCO, ...), with TF annotations from experimental evidence for a large subset.

The collection was clustered on pairwise motif similarity at several resolutions, and those clusters
annotated to TF families with [AnimalTFDB4](https://guolab.wchscu.cn/AnimalTFDB4/#/) — listed below
under `Annotation for cluster resolutions`. To keep the similarity step cheap, the collection is
downsampled to a fixed number of motifs per cluster (`PCA data for number of motifs per cluster`).

In [2]:
import tfmindi as tm

import os
import re
import numpy as np
from tqdm import tqdm

In [3]:
# Fetch the (clustered) motif collection.
motif_collection = tm.MotifCollectionData("mcv11.refdata.tar.gz")

In [4]:
# Sample n motifs per cluster; fewer motifs means a cheaper similarity step.
N_MOTIFS_PER_CLUSTER = 20
motif_collection_motifs = motif_collection.get_motifs(N_MOTIFS_PER_CLUSTER)

# Keys are (source_file, motif_name) tuples. Since the collection is a meta-database, a name either
# points back to its source database or, for a cluster's consensus motif, is a hash.
print("n sampled motifs:", len(motif_collection_motifs))
for (_, name), ppm in list(motif_collection_motifs.items())[:5]:
    print(f"  {name:<50} PPM {ppm.shape}")

n sampled motifs: 5707
  homer/homer__MCAGCTGBYH_Twist2.cb                  PPM (4, 10)
  homer/homer__YTAATTRAWWCCAGATGT_Pitx1_Ebox.cb      PPM (4, 18)
  0x7f3d98fff099fbe5                                 PPM (4, 20)
  homer/homer__TTCTAGAABNTTCTA_HRE.cb                PPM (4, 15)
  0x405fc587e9c161e8                                 PPM (4, 12)


### Clusters and TF families

Every motif carries a cluster assignment per resolution (`leiden_<resolution>` in the metadata), and
every cluster is annotated to a TF family. Higher resolution means more, finer clusters; we
annotate seqlets at `5.0` below.

In [5]:
metadata = motif_collection.metadata
resolutions = [c.removeprefix("leiden_") for c in metadata.columns if c.startswith("leiden_")]

print(f"{len(metadata)} motifs, clustered at {len(resolutions)} resolutions: {', '.join(resolutions)}")
for res in ("1.0", "3.0", "5.0"):
    print(f"  leiden_{res}: {metadata[f'leiden_{res}'].nunique()} clusters")

70041 motifs, clustered at 21 resolutions: 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0, 3.2, 3.4, 3.6, 3.8, 4.0, 4.2, 4.4, 4.6, 4.8, 5.0
  leiden_1.0: 115 clusters
  leiden_3.0: 210 clusters
  leiden_5.0: 296 clusters


In [6]:
# Candidate families per cluster, with the enrichment p-value that ranks them.
motif_collection.get_cluster_annotation("5.0").head()

,cluster,family,pval,pval_adj
0,0,bHLH,4.799394e-134,1.439818e-133
1,0,"bHLH, TEA",1.380815e-01,2.071222e-01
2,0,zf-C2H2,1.000000e+00,1.000000e+00
3,1,CUT,3.311586e-05,6.623172e-05
4,1,Homeobox,9.667333e-11,3.866933e-10


## Extracting seqlets

Seqlets are spans of nucleotides with high importance. {func}`~tfmindi.pp.extract_seqlets` projects
the contributions onto a 1D track (`(contrib * oh).sum(1)`) and applies a seqlet caller, selected
with `method`:

- `"recursive_q99_abs_smooth"` (**default**) — smooths the track, normalizes each region by its 99th
  percentile and calls on *absolute* importance, so activating and repressing seqlets are both captured.
- `"recursive_raw"` — the recursive caller on the raw signed track. Pre-v2 behaviour, based on
  `tangermeme`'s [recursive_seqlets](https://tangermeme.readthedocs.io/en/latest/tutorials/Tutorial_A4_Seqlets.html#Recursive-Seqlets);
  positive seqlets only.
- `"hysteresis"` — two-threshold local caller (high-importance seed, lower-importance growth).
- `"local_contrast"` — multi-scale sliding-window contrast caller.
- `"wavelet_otsu"` — wavelet denoising followed by Otsu thresholding.
- `"finemo_fit_contrib"` — fits motif CWMs against the raw contributions (requires `finemo`).

Hyperparameters are method-specific keyword arguments: `threshold=` and `additional_flanks=` for the
recursive callers, `seed_z=` for `"hysteresis"`, `threshold_scale=` for `"wavelet_otsu"`.

Inputs are `(n_regions, 4, width)`. Returned seqlets are scaled to [-1, 1] and sign-corrected so
their average contribution is positive.

In [7]:
# extract_seqlets expects (n_regions, 4, width), so concatenate the cell type specific contributions.
# Tracking which region came from which cell type is optional, but useful for interpretation later.
CONTRIB_FOLDER = "tutorial_data/modisco_results_ft_2000/"

contrib_list = []
oh_list = []
classes_list = []

class_names = [
    re.match(r"(.+?)_oh\.npz$", f).group(1)  # type: ignore
    for f in os.listdir(CONTRIB_FOLDER)
    if f.endswith("_oh.npz")
]

for i, c in enumerate(tqdm(class_names)):
    contrib_list.append(np.load(os.path.join(CONTRIB_FOLDER, f"{c}_contrib.npz"))["arr_0"])
    oh_list.append(np.load(os.path.join(CONTRIB_FOLDER, f"{c}_oh.npz"))["arr_0"])
    classes_list.append(np.repeat(c, oh_list[i].shape[0]))

oh = np.concatenate(oh_list)
contrib = np.concatenate(contrib_list)
classes = np.concatenate(classes_list)
region_id_to_ct_map = {i: str(c) for i, c in enumerate(classes)}

100%|██████████| 19/19 [00:20<00:00,  1.06s/it]


In [8]:
seqlets_df, seqlets_matrices = tm.pp.extract_seqlets(
    contrib=contrib,
    oh=oh,
    method="recursive_q99_abs_smooth",  # default caller
    threshold=0.05,  # lower = fewer seqlets
    additional_flanks=3,  # flanking bases to include around each seqlet
)

Processing seqlets: 100%|██████████| 582915/582915 [00:04<00:00, 136938.93it/s]


In [9]:
len(seqlets_matrices)

582915

In [10]:
seqlets_df.head(3)  # information on seqlet position and significance

,example_idx,start,end,attribution,score
0,0,1022,1047,11.616397,1.467698
1,0,1052,1069,8.770116,1.617959
2,0,1068,1086,8.682135,1.586178


## Calculating motif similarity

{func}`~tfmindi.pp.calculate_motif_similarity` runs TomTom between every seqlet and every motif in
the collection; the resulting p-values are log-transformed and negated. This is the most expensive
step of the pipeline, and the one that gains most from a GPU.

At genome scale the result is the largest object in the pipeline, tens of GB, because the default
`threshold` keeps most of each seqlet's profile. `n_nearest` stores only the strongest hits per
seqlet instead, but the complete profile is exactly what
{func}`~tfmindi.tl.predict_tf_family_seqlets` projects into the reference space.

Passing `reference` resolves that: each chunk is projected while its full profile is still in
memory, and only the pruned matrix is accumulated. The projection comes back alongside the matrix,
and annotation later reads it instead of `.X`.

```{note}
`chunk_size` bounds peak memory without changing the result, lower it if the kernel runs out.
Leave `reference` off and you get the plain matrix back, but you should never need this.
```

In [11]:
print("using backend:", tm.get_backend())

using backend: gpu


In [12]:
# chunk_size bounds peak memory and does not change the result; lower it if the kernel dies.
# reference= projects every full chunk before pruning, so n_nearest costs no accuracy.
sim_matrix, reference_projection = tm.pp.calculate_motif_similarity(
    seqlets_matrices,
    motif_collection_motifs,
    chunk_size=50000,
    n_nearest=100,  # how many similarities to *store* per seqlet
    reference=motif_collection,
    n_motifs_per_reference_cluster=N_MOTIFS_PER_CLUSTER,
)
print(sim_matrix.shape, reference_projection.shape)

Processing chunks: 100%|██████████| 12/12 [10:31<00:00, 52.63s/it]


(582915, 5707) (582915, 50)


Everything processed so far goes into a single AnnData, which is the input to every `tfmindi.tl`
and `tfmindi.pl` function. The motif collection is optional, but it is what enables the annotation
and logo plots later on.

In [13]:
adata = tm.pp.create_seqlet_adata(
    sim_matrix,  # mandatory
    seqlets_df,  # mandatory
    oh_sequences=oh,
    contrib_scores=contrib,
    motif_collection=motif_collection_motifs,
)
adata

AnnData object with n_obs × n_vars = 582915 × 5707
    obs: 'example_idx', 'start', 'end', 'attribution', 'score', 'example_oh_idx', 'example_contrib_idx'
    var: 'motif_ppm'
    uns: 'unique_examples'
    layers: None (.X)

In [14]:
# Hand the projection to the AnnData; predict_tf_family_seqlets picks it up from this key.
adata.obsm[f"X_ref_proj_{N_MOTIFS_PER_CLUSTER}"] = reference_projection

The object holds:

- `.X` — the sparse seqlet x motif similarity matrix. Only the `n_nearest` strongest hits per seqlet
  are kept: the full profile would cost tens of GB, and nothing downstream needs it — clustering
  fits its own PCA on whatever `.X` holds, and annotation reads the projection in `.obsm` instead.
- `.obs` — seqlet metadata (`example_idx`, `start`, `end`, `attribution`, `score`) plus
  `example_oh_idx` / `example_contrib_idx`, which index into `.uns["unique_examples"]`
- `.var` — motif names and their PPMs, plus any annotations you passed in
- `.obsm[f"X_ref_proj_{N_MOTIFS_PER_CLUSTER}"]` — the reference projection returned above
- `.uns["unique_examples"]` — the de-duplicated one-hot and contribution arrays of the source regions

Per-seqlet one-hot and contribution matrices are slices of `.uns["unique_examples"]`, not `.obs`
columns. Read them with {func}`~tfmindi.pp.get_seqlet_ohs` and {func}`~tfmindi.pp.get_seqlet_matrices`.

In [15]:
# Optional: map each seqlet back to the cell type its region came from.
# this mapping depends on how you stored your original attribution arrays, so adapt to your usecase
adata.obs["cell_type"] = adata.obs["example_idx"].map(region_id_to_ct_map).astype("category")

In [16]:
adata.obs.head(3)

,example_idx,start,end,attribution,score,example_oh_idx,example_contrib_idx,cell_type
0,0,1022,1047,11.616397,1.467698,0,0,Astro
1,0,1052,1069,8.770116,1.617959,0,0,Astro
2,0,1068,1086,8.682135,1.586178,0,0,Astro


In [17]:
adata.var.head(3)

,motif_ppm
homer/homer__MCAGCTGBYH_Twist2.cb,"[[0.36, 0.001, 0.997, 0.001, 0.358, 0.001, 0.0..."
homer/homer__YTAATTRAWWCCAGATGT_Pitx1_Ebox.cb,"[[0.221, 0.318, 0.694, 0.995, 0.147, 0.182, 0...."
0x7f3d98fff099fbe5,"[[0.173, 0.226, 0.288, 0.105, 0.334, 0.06, 0.0..."


## Predicting the TF family of each seqlet

Seqlets are projected into the motif collection's PCA space and classified against it with
{func}`~tfmindi.tl.predict_tf_family_seqlets`:

1. Each seqlet is projected into the reference PCA space.
2. Its k nearest reference motifs vote on a **cluster label** of the motif collection.
3. Cluster labels carry a TF family annotation.

This is the step that needs the complete similarity profile computed above.

In [18]:
res = 5.0  # cluster resolution of the TF-MINDI motif collection
tm.tl.predict_tf_family_seqlets(
    adata=adata,
    motif_collection=motif_collection,
    cluster_resolution=res,
    n_motifs_per_reference_cluster=N_MOTIFS_PER_CLUSTER,
)

Reusing existing reference projection in obsm['X_ref_proj_20'] ...
building index ...
predicting ...
Added following keys to adata.obs: 
	predicted_5.0_predicted_cluster
	predicted_5.0_predicted_cluster_score
	predicted_5.0_predicted_family


Each seqlet is annotated to a TF-MINDI motif collection cluster.
Those clusters are annotated to TF-families.

In [19]:
adata.obs[
    [
        f"predicted_{res}_predicted_cluster",
        f"predicted_{res}_predicted_cluster_score",
        f"predicted_{res}_predicted_family",
    ]
]

,predicted_5.0_predicted_cluster,predicted_5.0_predicted_cluster_score,predicted_5.0_predicted_family
0,41,0.666667,41|CTF_NFI
1,41,0.933333,41|CTF_NFI
2,41,1.000000,41|CTF_NFI
3,131,1.000000,131|bHLH
4,172,1.000000,172|Homeobox
...,...,...,...
582910,88,1.000000,88|T-box
582911,105,0.600000,105|zf-C2H2
582912,81,0.266667,81|Homeobox
582913,59,0.533333,59|Homeobox


Family annotations are formatted as `<cluster>|<family>`, because a single TF family can still hold
quite some motif diversity; the cluster captures it.  
For plotting we also add a column with the family alone.

In [20]:
adata.obs[f"predicted_{res}_predicted_family_no_cl"] = [
    x.split("|")[1] if "|" in x else x for x in adata.obs[f"predicted_{res}_predicted_family"]
]

Because the projection was built from the complete profiles, this annotation is identical to what a
full similarity matrix would have produced while `.X` stores only 100 similarities per seqlet.
The projection is cached in `.obsm` and does not depend on the resolution, so re-annotating at
another `cluster_resolution` reuses it and never reads `.X` again.

## Saving our preprocessed data

Use {func}`~tfmindi.save_h5ad` and {func}`~tfmindi.load_h5ad` rather than AnnData's own writer: the
object stores numpy arrays in `.var` and `/`-containing keys in `.uns`, neither of which plain
`write_h5ad` accepts. Ours are thin wrappers that move those around and restore them on load.
Be aware that these files can get large.

In [21]:
tm.save_h5ad(adata, "tutorial_data/seqlets.h5ad")

... storing 'predicted_5.0_predicted_family' as categorical
... storing 'predicted_5.0_predicted_family_no_cl' as categorical
